# Test 3 — Selective Coref + Entity-Rich Retrieval (DAPR NaturalQuestions, local)

**Motivation (from Test 2):** On DAPR ConditionalQA, coref-before-embed was **flat** and the equal-weight
hybrid was **worse**. The worked example showed why: that corpus's coreference resolves to **common
nouns** ("them" -> "certain mobility aids"), which add no searchable signal. Test 1 (biography text)
*did* benefit, because coref injected **named entities** the query searched for ("he" -> "Marcel Mauss").

**Hypothesis under test here:**
> Coref-before-embed helps retrieval **only when it injects a named entity a query searches for** —
> not when it resolves a pronoun to a common noun.

**Two changes vs Test 2, to test this cleanly:**

| Change | Test 2 | Test 3 |
|--------|--------|--------|
| **Coref scope** | rewrite *every* pronoun | **selective**: rewrite only when the antecedent is a **proper noun** (named entity) |
| **Dataset** | ConditionalQA (gov guidance, common-noun coref) | **DAPR NaturalQuestions** (Wikipedia, entity-rich coref) |
| **Hybrid fusion** | equal-weight RRF | **weighted RRF** (dense 0.7 / lexical 0.3) — fixes Test 2's hybrid regression on long queries |

Everything else is identical to Test 2 (bge-small dense + BM25, official qrels, document-level coref,
disk cache, no re-chunking, runs on a free Kaggle T4).

## Variants

| Label | Retrieval |
|-------|-----------|
| **baseline** | Dense on original passage text |
| **coref_dense** | Dense on the **selective** LingMess rewrite (proper-noun antecedents only) |
| **coref_hybrid** | **Weighted RRF**: 0.3 * BM25(original) + 0.7 * dense(coref) |

**Invariants:** original text is always returned to the user; gold = official qrels (`score>0`);
same passage IDs across variants; 1:1 coref alignment asserted.


## 1. Setup & Install

Same stack as Test 2. `fastcoref` + pinned `transformers<5` for LingMess (transformers 4.x API).

In [ ]:
!pip install -q datasets pytrec_eval tqdm pandas numpy rank_bm25 sentence-transformers
!pip install -q "transformers<5" fastcoref
!python -m spacy download en_core_web_sm
# IMPORTANT: after this cell finishes, RESTART THE KERNEL once so pinned transformers 4.x is the
# version actually imported, then run the cells below (skip re-running this one).

## 2. Config

`SELECTIVE_COREF=True` is the key knob for Test 3. `DATASET="NaturalQuestions"` is the entity-rich
DAPR split (Wikipedia). The NQ corpus is large; `load_dataset` memory-maps it (low RAM), but the
download is a few GB and we cap the working pool + subsample queries to fit a free T4.

In [ ]:
import os, logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("coref_v3")
logging.getLogger("fastcoref").setLevel(logging.WARNING)   # silence per-call coref text logs

# The "Map:" progress bars during coref come from the `datasets` library (fastcoref tokenizes via
# datasets.map). Disable them so they don't flood / freeze the Kaggle browser tab.
try:
    import datasets
    datasets.disable_progress_bars()
except Exception:
    pass

# --- Model (unchanged from Test 2) ---
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
BATCH_SIZE  = 64
MAX_LEN     = 512

# --- Test-3 experiment knobs ---
SELECTIVE_COREF = True          # rewrite a pronoun ONLY if its antecedent is a proper noun (named entity)
DATASET         = "NaturalQuestions"   # entity-rich DAPR split (Wikipedia)
DENSE_WEIGHT    = 0.7           # weighted-RRF weight for dense(coref); lexical(BM25) gets (1 - this)

# --- Core config ---
TOP_K       = 5
RRF_K       = 60
CACHE_DIR   = "./embed_cache_v3"
RUN_DEPTH   = 100
COREF_WINDOW_WORDS = 350
COREF_CHUNK        = 512        # windows per batched LingMess predict() call (GPU-efficient)

# --- Subsampling for a free-T4 POC (documented in findings) ---
# NQ gold docs are full Wikipedia articles (~37 passages each), so #passages ~= 37 * #gold-docs.
# 200 queries -> ~200 gold docs -> a few thousand passages. Raise later if you have compute.
MAX_QUERIES  = 200
CORPUS_MAX   = 8000             # document-aware cap (whole gold-containing docs first, then top up)

os.makedirs(CACHE_DIR, exist_ok=True)
log.info("Config | model=%s | dataset=%s | selective_coref=%s | dense_w=%.2f | cap=%d | max_q=%d",
         EMBED_MODEL, DATASET, SELECTIVE_COREF, DENSE_WEIGHT, CORPUS_MAX, MAX_QUERIES)

## 3. Coreference engine (LingMess) with a SELECTIVE switch

Reused from Test 1/2, with one addition: `_pick_representative(..., selective=)`. When
`SELECTIVE_COREF=True`, a cluster is only rewritten if its representative is a **proper noun**
(named entity). Common-noun antecedents (e.g. "certain mobility aids") are **left as pronouns** — so
we never inject a redundant common noun, isolating the entity-injection effect.

Proper-noun detection is a heuristic: a token with an uppercase initial, not all-caps, length >= 2,
and not a common capitalized function word (to reduce sentence-initial false positives). Not perfect,
but good enough to separate "named entity" from "common noun" antecedents.

In [ ]:
import re, time
SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")
PRONOUN_RE = re.compile(r"\b(he|she|him|her|his|hers|they|them|their|theirs|it|its)\b", re.IGNORECASE)

def split_sentences(text):
    return [s.strip() for s in SENT_SPLIT_RE.split(text.strip()) if s.strip()]

from transformers.models.auto.modeling_auto import AutoModel
if not getattr(AutoModel, "_coref_eager_patched", False):
    _orig_from_config = AutoModel.from_config.__func__
    @classmethod
    def _from_config_eager(cls, config, **kwargs):
        try:
            config._attn_implementation = "eager"
        except Exception:
            pass
        kwargs.setdefault("attn_implementation", "eager")
        try:
            return _orig_from_config(cls, config, **kwargs)
        except (TypeError, ValueError):
            kwargs.pop("attn_implementation", None)
            return _orig_from_config(cls, config, **kwargs)
    AutoModel.from_config = _from_config_eager
    AutoModel._coref_eager_patched = True

from fastcoref import LingMessCoref
_has_cuda = False
try:
    import torch
    _has_cuda = torch.cuda.is_available()
except Exception:
    pass
coref_model = LingMessCoref(device="cuda" if _has_cuda else "cpu")
log.info("LingMessCoref loaded | device=%s | selective=%s", "cuda" if _has_cuda else "cpu", SELECTIVE_COREF)

PRONOUN_SET = {"he","she","him","her","his","hers","they","them","their","theirs","it","its",
               "himself","herself","themselves","itself"}
# Common capitalized words that appear at sentence start but are NOT named entities.
COMMON_CAP_WORDS = {"the","a","an","this","that","these","those","if","when","you","your","yours",
                    "it","he","she","they","we","i","in","on","at","for","to","and","but","or",
                    "there","here","what","which","who","how","why","as","is","are","was","were"}

def _is_pronoun_span(text):
    return text.strip().lower() in PRONOUN_SET

def _has_pronoun_token(text):
    return any(w.strip(".,;:'\"").lower() in PRONOUN_SET for w in text.split())

def _has_proper_noun(text):
    """Heuristic: at least one token looks like a named-entity word."""
    for t in text.split():
        w = t.strip(".,;:'\"()[]")
        if len(w) >= 2 and w[0].isupper() and not w.isupper() and w.lower() not in COMMON_CAP_WORDS:
            return True
    return False

def _pick_representative(spans_text, selective=False):
    """Choose the cluster's replacement name.
    selective=True -> only accept a PROPER-NOUN representative (else return None => skip cluster)."""
    proper = [s for s in spans_text if not _has_pronoun_token(s) and _has_proper_noun(s)]
    clean = [s for s in spans_text if not _has_pronoun_token(s)]
    pool = proper if selective else (proper or clean)
    if not pool:
        return None
    return min(pool, key=len).strip()

def clusters_to_edits(text, clusters, selective):
    """Turn coref clusters (char-offset spans) into pronoun-replacement edits for `text`."""
    edits = []
    for cluster in clusters:
        spans_text = [text[s:e] for s, e in cluster]
        rep = _pick_representative(spans_text, selective=selective)
        if not rep:
            continue
        for (s, e), st in zip(cluster, spans_text):
            if _is_pronoun_span(st) and st.strip().lower() != rep.lower():
                repl = rep + "'s" if st.strip().lower() in {"his","her","its","their","hers","theirs"} else rep
                edits.append((s, e, repl))
    return sorted(edits, key=lambda x: x[0])

def coref_edits(text, selective=None):
    """Single-text convenience wrapper (used by the demo + singleton passages)."""
    if selective is None:
        selective = SELECTIVE_COREF
    if not text.strip():
        return []
    preds = coref_model.predict(texts=[text])
    if not preds:
        return []
    return clusters_to_edits(text, preds[0].get_clusters(as_strings=False), selective)

def apply_edits(text, edits):
    out = text
    for s, e, repl in sorted(edits, key=lambda x: x[0], reverse=True):
        out = out[:s] + repl + out[e:]
    return out

def resolve_coref_text(text):
    return apply_edits(text, coref_edits(text))

# Quick sanity check of the selective gate on hand-written snippets.
_demo = [
    "Marie Curie won the Nobel Prize. She later won a second one.",          # proper -> rewrite
    "You pay VAT on certain mobility aids when you pay for them at home.",    # common noun -> skip (selective)
]
for s in _demo:
    print("ORIG    :", s)
    print("SELECTIVE:", resolve_coref_text(s))
    print("-" * 70)

### Document-level coref — batched (see Section 8)

Coref runs at the **document level** (a pronoun's antecedent may live in an earlier passage): passages of a doc are packed into word-budgeted windows and char-offset edits are routed back to the owning passage (1:1, no re-chunking). For speed on Wikipedia-length docs, the actual work is **batched across the whole corpus** in Section 8 (`build_coref_texts`) rather than one `predict()` per window. The helper below is kept for reference / single-doc use.

In [ ]:
# Reference implementation (single-doc, unbatched). The pipeline uses the BATCHED
# corpus-wide version in Section 8 (build_coref_texts) for speed; this is kept for clarity.
def coref_passages_doc_level(passages, window_words=COREF_WINDOW_WORDS):
    texts = [p["text"] for p in passages]
    out = list(texts)
    windows, cur, cur_w = [], [], 0
    for i, t in enumerate(texts):
        w = len(t.split())
        if cur and cur_w + w > window_words:
            windows.append(cur); cur, cur_w = [], 0
        cur.append(i); cur_w += w
    if cur:
        windows.append(cur)
    for win in windows:
        spans, pos, parts = [], 0, []
        for k, idx in enumerate(win):
            if k > 0:
                parts.append(" "); pos += 1
            start = pos
            parts.append(texts[idx]); pos += len(texts[idx])
            spans.append((idx, start, pos))
        joined = "".join(parts)
        edits = coref_edits(joined)
        by_p = {}
        for (s, e, repl) in edits:
            for (idx, ss, se) in spans:
                if ss <= s and e <= se:
                    by_p.setdefault(idx, []).append((s - ss, e - ss, repl)); break
        for idx, local in by_p.items():
            out[idx] = apply_edits(texts[idx], local)
    return out

## 4. Encoder (bge-small dense) with disk cache (unchanged from Test 2)

In [ ]:
import hashlib, pickle
import numpy as np
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

def _hash_texts(texts):
    h = hashlib.sha1()
    for t in texts:
        h.update(t.encode("utf-8", "ignore")); h.update(b"\x00")
    return h.hexdigest()[:16]

def _cache_file(dataset, variant, role, texts):
    return os.path.join(CACHE_DIR, f"{dataset}__{variant}__{role}__{_hash_texts(texts)}.pkl")

dense_model = SentenceTransformer(EMBED_MODEL, device="cuda" if _has_cuda else "cpu")
log.info("Loaded %s (dense) + rank_bm25 (lexical)", EMBED_MODEL)

def encode_cached(dataset, variant, role, texts):
    cf = _cache_file(dataset, variant, role, texts)
    if os.path.exists(cf):
        with open(cf, "rb") as f:
            arr = pickle.load(f)
        log.info("cache HIT  | %s", os.path.basename(cf))
        return arr
    log.info("cache MISS | encoding %d %s texts (%s/%s)", len(texts), role, dataset, variant)
    arr = dense_model.encode(texts, batch_size=BATCH_SIZE, normalize_embeddings=True,
                             show_progress_bar=False).astype("float32")
    with open(cf, "wb") as f:
        pickle.dump(arr, f)
    return arr

## 5. Retrieval: dense, BM25, and **weighted** RRF

Weighted RRF: `score(p) = DENSE_WEIGHT * 1/(k+rank_dense) + (1-DENSE_WEIGHT) * 1/(k+rank_bm25)`.
At `DENSE_WEIGHT=0.7` the strong dense signal leads and BM25 refines — addressing Test 2's
equal-weight hybrid regression on long queries.

In [ ]:
def dense_ranklist(q_dense, p_dense, depth=RUN_DEPTH):
    sims = q_dense @ p_dense.T
    order = np.argsort(-sims, axis=1)[:, :depth]
    return [[(int(j), float(sims[qi, j])) for j in order[qi]] for qi in range(sims.shape[0])]

def bm25_ranklist(query_texts, passage_texts, depth=RUN_DEPTH):
    from rank_bm25 import BM25Okapi
    tok = lambda s: re.findall(r"[a-z0-9]+", s.lower())
    bm25 = BM25Okapi([tok(t) for t in passage_texts])
    out = []
    for q in tqdm(query_texts, desc="BM25", mininterval=2.0):
        scores = bm25.get_scores(tok(q))
        order = np.argsort(-scores)[:depth]
        out.append([(int(j), float(scores[j])) for j in order])
    return out

def weighted_rrf(dense_rl, bm25_rl, dense_w=DENSE_WEIGHT, k=RRF_K, depth=RUN_DEPTH):
    """Weighted reciprocal-rank fusion of dense(coref) and BM25(original)."""
    nq = len(dense_rl)
    w = {"dense": dense_w, "bm25": 1.0 - dense_w}
    fused = []
    for qi in range(nq):
        acc = {}
        for sysname, rl in (("dense", dense_rl), ("bm25", bm25_rl)):
            for rank, (pidx, _s) in enumerate(rl[qi]):
                acc[pidx] = acc.get(pidx, 0.0) + w[sysname] * 1.0 / (k + rank + 1)
        ordered = sorted(acc.items(), key=lambda x: -x[1])[:depth]
        fused.append([(int(p), float(s)) for p, s in ordered])
    return fused

## 6. Metrics (official qrels, `pytrec_eval`) — unchanged from Test 2

In [ ]:
import pytrec_eval

def build_run(ranklist, query_ids, passage_ids):
    run = {}
    for qi, qid in enumerate(query_ids):
        run[qid] = {passage_ids[pidx]: float(score) for pidx, score in ranklist[qi]}
    return run

def eval_run(run, qrels, k=TOP_K):
    ev = pytrec_eval.RelevanceEvaluator(qrels, {"recall_5", "ndcg_cut_10", "recip_rank"})
    per = ev.evaluate(run)
    recall = float(np.mean([v["recall_5"] for v in per.values()])) if per else 0.0
    ndcg = float(np.mean([v["ndcg_cut_10"] for v in per.values()])) if per else 0.0
    mrr = float(np.mean([v["recip_rank"] for v in per.values()])) if per else 0.0
    gold = {q: {d for d, s in rels.items() if s > 0} for q, rels in qrels.items()}
    hits = set()
    for qid, docs in run.items():
        topk = [d for d, _ in sorted(docs.items(), key=lambda x: -x[1])[:k]]
        if gold.get(qid) and (set(topk) & gold[qid]):
            hits.add(qid)
    return {"recall_at_5": recall, "ndcg_at_10": ndcg, "mrr": mrr, "hits": hits, "n": len(qrels)}

## 7. Dataset loader — DAPR NaturalQuestions (entity-rich Wikipedia)

Same loader shape as Test 2. We subsample to `MAX_QUERIES` test queries, then apply the
document-aware cap (whole gold-containing documents first, then top up to `CORPUS_MAX`). This keeps
the hard same-document distractors that make retrieval real, while fitting a free T4.

In [ ]:
from datasets import load_dataset

def load_dapr(dataset_name=DATASET, cap=CORPUS_MAX, max_queries=MAX_QUERIES):
    """Memory-safe + FAST DAPR loader for large corpora (NQ = 2.68M passages).

    Avoids two traps:
      * never materialize the 773MB `text` column for the whole corpus, and
      * never loop over 2.68M rows in pure Python (that hangs).
    Strategy: read only the light id/doc columns into a pandas DataFrame and filter with
    vectorized `isin` (C-speed), then `select` + fetch text for just the ~cap kept rows.
    """
    corpus = load_dataset("UKPLab/dapr", f"{dataset_name}-corpus", split="test")
    queries_ds = load_dataset("UKPLab/dapr", f"{dataset_name}-queries", split="test")
    qrels_ds = load_dataset("UKPLab/dapr", f"{dataset_name}-qrels", split="test")

    qtext = {str(q["_id"]): (q["text"] or "") for q in queries_ds}
    qrels_all = {}
    for r in qrels_ds:
        if int(r["score"]) > 0:
            qrels_all.setdefault(str(r["query_id"]), {})[str(r["corpus_id"])] = int(r["score"])

    # Subsample queries (deterministic: first max_queries with text + qrels).
    qids = [q for q in qrels_all if q in qtext][:max_queries]
    queries = {q: qtext[q] for q in qids}
    qrels = {q: qrels_all[q] for q in qids}
    gold_ids = {pid for rels in qrels.values() for pid, s in rels.items() if s > 0}

    # ---- Light scan: only id/doc/paragraph columns -> pandas (drops the huge text col). ----
    light_cols = [c for c in ["_id", "doc_id", "paragraph_no"] if c in corpus.column_names]
    t0 = time.time()
    df = corpus.select_columns(light_cols).to_pandas()
    df["_id"] = df["_id"].astype(str)
    if "doc_id" in df.columns:
        df["doc_id"] = df["doc_id"].astype(str)
    else:
        df["doc_id"] = df["_id"]
    if "paragraph_no" not in df.columns:
        df["paragraph_no"] = 0
    log.info("[DAPR %s] light scan: %d passages in %.1fs", dataset_name, len(df), time.time() - t0)

    # ---- Vectorized doc-aware cap (gold docs first, then top up to cap). ----
    gold_doc_ids = set(df.loc[df["_id"].isin(gold_ids), "doc_id"].unique())
    is_gold_doc = df["doc_id"].isin(gold_doc_ids)
    gold_idx = df.index[is_gold_doc].tolist()               # gold docs in full (same-doc distractors)
    keep_idx = list(gold_idx)
    if len(keep_idx) < cap:
        other_idx = df.index[~is_gold_doc].tolist()[: cap - len(keep_idx)]
        keep_idx.extend(other_idx)
    log.info("[DAPR %s] corpus=%d | gold passages=%d | gold docs=%d | keeping=%d",
             dataset_name, len(df), len(gold_ids), len(gold_doc_ids), len(keep_idx))

    # ---- Fetch text ONLY for kept rows (one Arrow select over ~cap rows). ----
    sub = corpus.select(keep_idx)
    sub_text = sub["text"]
    sub_id = df.loc[keep_idx, "_id"].tolist()
    sub_doc = df.loc[keep_idx, "doc_id"].tolist()
    sub_par = df.loc[keep_idx, "paragraph_no"].tolist()
    passages = []
    for j in range(len(keep_idx)):
        par = sub_par[j]
        passages.append({
            "_id": sub_id[j],
            "text": (sub_text[j] or ""),
            "doc_id": sub_doc[j],
            "order": int(par) if par is not None else 0,
        })

    log.info("[DAPR %s] kept=%d passages across %d docs (cap=%d) | queries=%d",
             dataset_name, len(passages), len({p["doc_id"] for p in passages}), cap, len(queries))
    return {"name": f"dapr_{dataset_name}", "passages": passages, "queries": queries, "qrels": qrels}

## 8. Build selective coref rewrites (1:1 with passage IDs)

Document-level coref with the **selective** gate active. We log pronoun reduction AND how many
passages were actually changed — with selective coref, far fewer passages change (only those whose
pronouns resolve to named entities), which is exactly the point.

In [ ]:
def _pack_windows(plist, window_words=COREF_WINDOW_WORDS):
    """Pack a document's passages into word-budgeted windows. Returns list of windows,
    each a list of passage dicts (order preserved)."""
    windows, cur, cur_w = [], [], 0
    for p in plist:
        w = len(p["text"].split())
        if cur and cur_w + w > window_words:
            windows.append(cur); cur, cur_w = [], 0
        cur.append(p); cur_w += w
    if cur:
        windows.append(cur)
    return windows

def build_coref_texts(ds):
    """Document-level SELECTIVE coref, batched across the WHOLE corpus for GPU efficiency.

    Instead of one predict() per window (slow, batch-size-1), we collect every window across all
    documents and run LingMess in batched chunks of COREF_CHUNK. Edits are routed back to the owning
    passage (1:1, official boundaries preserved). Cached to disk so coref runs once."""
    tag = "sel" if SELECTIVE_COREF else "full"
    cache = os.path.join(CACHE_DIR, f"coref_{tag}__{ds['name']}.pkl")
    if os.path.exists(cache):
        with open(cache, "rb") as f:
            mapping = pickle.load(f)
        for p in ds["passages"]:
            p["coref_text"] = mapping.get(p["_id"], p["text"])
        log.info("[%s] coref cache HIT (%d passages, %s)", ds["name"], len(mapping), tag)
    else:
        from collections import defaultdict
        groups = defaultdict(list)
        for p in ds["passages"]:
            groups[p["doc_id"]].append(p)
        for p in ds["passages"]:
            p["coref_text"] = p["text"]           # default: unchanged

        # ---- 1) Build all windows across all docs (record char-span -> passage _id). ----
        all_texts, all_spans = [], []
        for did, plist in groups.items():
            plist.sort(key=lambda x: x["order"])
            for win in _pack_windows(plist):
                spans, pos, parts = [], 0, []
                for k, p in enumerate(win):
                    if k > 0:
                        parts.append(" "); pos += 1
                    start = pos
                    parts.append(p["text"]); pos += len(p["text"])
                    spans.append((p["_id"], start, pos))
                all_texts.append("".join(parts)); all_spans.append(spans)
        log.info("[%s] coref: %d windows over %d docs / %d passages",
                 ds["name"], len(all_texts), len(groups), len(ds["passages"]))

        # ---- 2) Batched LingMess predict in chunks (one progress bar, GPU-efficient). ----
        t0 = time.time()
        edits_by_pid = defaultdict(list)
        byid = {p["_id"]: p for p in ds["passages"]}
        for i in tqdm(range(0, len(all_texts), COREF_CHUNK), desc=f"coref[{tag}] {ds['name']}", mininterval=2.0):
            batch_texts = all_texts[i:i + COREF_CHUNK]
            batch_spans = all_spans[i:i + COREF_CHUNK]
            preds = coref_model.predict(texts=batch_texts)
            for text, spans, pred in zip(batch_texts, batch_spans, preds):
                edits = clusters_to_edits(text, pred.get_clusters(as_strings=False), SELECTIVE_COREF)
                for (s, e, repl) in edits:
                    for (pid, ss, se) in spans:
                        if ss <= s and e <= se:
                            edits_by_pid[pid].append((s - ss, e - ss, repl)); break

        # ---- 3) Apply edits per passage. ----
        for pid, local in edits_by_pid.items():
            byid[pid]["coref_text"] = apply_edits(byid[pid]["text"], local)

        mapping = {p["_id"]: p["coref_text"] for p in ds["passages"]}
        with open(cache, "wb") as f:
            pickle.dump(mapping, f)
        log.info("[%s] coref built | %.1fs | %d passages (%s)", ds["name"], time.time()-t0, len(mapping), tag)

    assert all("coref_text" in p for p in ds["passages"]), "coref alignment broken"
    changed = sum(1 for p in ds["passages"] if p["coref_text"] != p["text"])
    orig_all = " ".join(p["text"] for p in ds["passages"])
    coref_all = " ".join(p["coref_text"] for p in ds["passages"])
    pb, pa = len(PRONOUN_RE.findall(orig_all)), len(PRONOUN_RE.findall(coref_all))
    log.info("[%s] passages changed by coref: %d/%d | pronouns %d -> %d (%.0f%% reduction)",
             ds["name"], changed, len(ds["passages"]), pb, pa, 100*(1-pa/max(pb,1)))
    return {"pron_before": pb, "pron_after": pa, "changed": changed, "n": len(ds["passages"])}

## 9. Run the three variants

In [ ]:
def run_dataset(ds):
    name = ds["name"]
    passages = ds["passages"]
    p_ids = [p["_id"] for p in passages]
    q_ids = list(ds["queries"].keys())
    q_texts = [ds["queries"][q] for q in q_ids]
    orig_texts = [p["text"] for p in passages]
    coref_texts = [p["coref_text"] for p in passages]

    q_dense       = encode_cached(name, "query", "query", q_texts)
    p_dense_orig  = encode_cached(name, "baseline", "passage", orig_texts)
    p_dense_coref = encode_cached(name, "coref_dense_" + ("sel" if SELECTIVE_COREF else "full"), "passage", coref_texts)

    rl_baseline = dense_ranklist(q_dense, p_dense_orig)
    rl_corefden = dense_ranklist(q_dense, p_dense_coref)
    rl_bm25     = bm25_ranklist(q_texts, orig_texts)
    rl_hybrid   = weighted_rrf(rl_corefden, rl_bm25)         # 0.7*dense(coref) + 0.3*BM25(orig)

    variants = {"baseline": rl_baseline, "coref_dense": rl_corefden, "coref_hybrid": rl_hybrid}
    results, runs = {}, {}
    for label, rl in variants.items():
        run = build_run(rl, q_ids, p_ids)
        runs[label] = run
        results[label] = eval_run(run, ds["qrels"])
        r = results[label]
        log.info("[%s/%s] R@5=%.3f nDCG@10=%.3f MRR=%.3f", name, label, r["recall_at_5"], r["ndcg_at_10"], r["mrr"])
    return {"name": name, "results": results, "runs": runs,
            "n_passages": len(passages), "n_queries": len(q_ids)}

## 10. Results table + flip analysis

In [ ]:
import pandas as pd

def summarize(run_out):
    res = run_out["results"]; base = res["baseline"]
    rows = []
    for label in ["baseline", "coref_dense", "coref_hybrid"]:
        r = res[label]
        rows.append({"variant": label, "recall@5": round(r["recall_at_5"],4),
                     "nDCG@10": round(r["ndcg_at_10"],4), "MRR": round(r["mrr"],4),
                     "Δrecall@5": round(r["recall_at_5"]-base["recall_at_5"],4),
                     "ΔnDCG@10": round(r["ndcg_at_10"]-base["ndcg_at_10"],4)})
    df = pd.DataFrame(rows)
    print(f"\n=== {run_out['name']} | passages={run_out['n_passages']} queries={run_out['n_queries']} ===")
    print(df.to_string(index=False))
    return df

def flips(run_out):
    res = run_out["results"]; base_hits = res["baseline"]["hits"]
    all_qids = set(run_out["runs"]["baseline"].keys()); out = {}
    for label in ["coref_dense", "coref_hybrid"]:
        h = res[label]["hits"]
        rec, hurt, bf = h - base_hits, base_hits - h, all_qids - base_hits - h
        out[label] = {"recovered": len(rec), "hurt": len(hurt), "both_fail": len(bf),
                      "recovered_ids": list(rec)[:10], "hurt_ids": list(hurt)[:10]}
        print(f"\n[{run_out['name']}] {label} vs baseline: recovered={len(rec)} hurt={len(hurt)} both_fail={len(bf)}")
    return out

## 11. Run — DAPR NaturalQuestions (all 3 variants)

Entity-rich Wikipedia. First run downloads the NQ corpus (a few GB) and builds selective coref;
cached afterward. Watch for the `coref built` and `passages changed by coref` lines.

In [ ]:
ALL_RUNS = {}; COREF_STATS = {}
ds = load_dapr(DATASET, cap=CORPUS_MAX, max_queries=MAX_QUERIES)
COREF_STATS[ds["name"]] = build_coref_texts(ds)
run_out = run_dataset(ds)
ALL_RUNS[ds["name"]] = run_out
df = summarize(run_out)
fl = flips(run_out)

## 12. Worked example — an entity-injection recovery case

Find a query where `coref_dense` recovered a baseline miss and show the gold passage's original vs
selective-coref rewrite. Here we expect the rewrite to inject a **named entity** (not a common noun),
which is the whole point of Test 3.

In [ ]:
def show_worked_example(ds, run_out, variant="coref_dense"):
    res = run_out["results"]
    recovered = res[variant]["hits"] - res["baseline"]["hits"]
    if not recovered:
        print(f"No recovery for {variant} on {run_out['name']}."); return
    by_id = {p["_id"]: p for p in ds["passages"]}
    shown = 0
    for qid in sorted(recovered):
        gold_ids = {pid for pid, s in ds["qrels"][qid].items() if s > 0}
        for gid in gold_ids:
            p = by_id.get(gid)
            if p and p["coref_text"] != p["text"]:      # prefer an example where coref actually changed text
                print("QUERY:", ds["queries"][qid])
                print("\n--- Gold passage", gid, "(recovered by", variant + ") ---")
                print("RETURNED (original):", p["text"][:500])
                print("\nEMBEDDED (selective coref):", p["coref_text"][:500])
                print("\npronouns  orig:", len(PRONOUN_RE.findall(p["text"])),
                      "| coref:", len(PRONOUN_RE.findall(p["coref_text"])))
                shown += 1
                break
        if shown:
            break
    if not shown:
        print("Recovered queries exist but none had a changed gold passage; showing first recovered qid.")
        qid = sorted(recovered)[0]; print("QUERY:", ds["queries"][qid])

show_worked_example(ds, ALL_RUNS[ds["name"]])

## 13. Write `test-3-findings.md`

In [ ]:
def _md_table(df):
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join("---" for _ in cols) + " |"]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join(str(r[c]) for c in cols) + " |")
    return "\n".join(lines)

def write_findings(path="test-3-findings.md"):
    out = []
    out.append("# Test 3 — Selective Coref + Entity-Rich Retrieval (DAPR NaturalQuestions)\n")
    out.append(f"**Run date:** {time.strftime('%Y-%m-%d')}  ")
    out.append("**Notebook:** `coref_public_eval_v3.ipynb`  ")
    out.append("**Hypothesis:** coref-before-embed helps retrieval only when it injects a named "
               "entity a query searches for (not a common noun).\n")
    out.append("## What changed vs Test 2\n")
    out.append("- **Selective coref:** rewrite a pronoun only when its antecedent is a **proper noun** "
               "(named entity); common-noun antecedents are left untouched.")
    out.append(f"- **Entity-rich dataset:** DAPR {DATASET} (Wikipedia), instead of gov-guidance ConditionalQA.")
    out.append(f"- **Weighted RRF hybrid:** {DENSE_WEIGHT:.1f} * dense(coref) + {1-DENSE_WEIGHT:.1f} * BM25(original) "
               "(fixes Test 2's equal-weight hybrid regression).\n")
    out.append("## Setup\n")
    out.append(f"- **Embedding model:** {EMBED_MODEL} (dense) + rank_bm25 (lexical), local GPU")
    out.append(f"- **Coref:** fastcoref LingMess, document-level, SELECTIVE_COREF={SELECTIVE_COREF}")
    out.append(f"- **Metrics:** Recall@5, nDCG@10, MRR (pytrec_eval); TOP_K={TOP_K}, RRF_K={RRF_K}")
    out.append(f"- **Subsampling (free-T4 POC):** first {MAX_QUERIES} test queries; document-aware corpus cap <= {CORPUS_MAX} passages.\n")

    for name, ro in ALL_RUNS.items():
        df = summarize(ro)
        out.append(f"## {name}\n")
        out.append(f"Passages: {ro['n_passages']} | Queries: {ro['n_queries']}")
        cs = COREF_STATS.get(name, {})
        if cs:
            out.append(f"Passages changed by selective coref: {cs['changed']}/{cs['n']} | "
                       f"pronouns {cs['pron_before']} -> {cs['pron_after']} "
                       f"({100*(1-cs['pron_after']/max(cs['pron_before'],1)):.0f}% reduction)\n")
        out.append(_md_table(df) + "\n")
        base_hits = ro["results"]["baseline"]["hits"]; all_qids = set(ro["runs"]["baseline"].keys())
        out.append("**Flips vs baseline**\n")
        out.append("| variant | recovered | hurt | both_fail |")
        out.append("| --- | --- | --- | --- |")
        for label in ["coref_dense", "coref_hybrid"]:
            h = ro["results"][label]["hits"]
            out.append(f"| {label} | {len(h-base_hits)} | {len(base_hits-h)} | {len(all_qids-base_hits-h)} |")
        out.append("")

    out.append("## How to read this\n")
    out.append("- If `coref_dense` > baseline here (while it was flat in Test 2), the hypothesis holds: "
               "coref helps when antecedents are named entities.")
    out.append("- `changed by selective coref` should be much smaller than a full-coref run — selective "
               "rewriting only touches entity-resolving pronouns, removing the common-noun noise seen in Test 2.")
    out.append(f"- Weighted RRF ({DENSE_WEIGHT:.1f}/{1-DENSE_WEIGHT:.1f}) should keep the hybrid at or above the "
               "dense baseline, unlike Test 2's equal-weight hybrid.")
    out.append("\n## Scope notes\n")
    out.append(f"- DAPR {DATASET} subsampled for a free-T4 POC (first {MAX_QUERIES} queries, corpus cap "
               f"<= {CORPUS_MAX}, whole gold-containing docs first). A subset, not the full corpus.")
    out.append("- Original text always returned to the user; official qrels only; 1:1 coref alignment asserted.")
    out.append("- Relative comparison across variants matters more than absolute SOTA numbers.")

    text = "\n".join(out) + "\n"
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    log.info("Wrote %s (%d chars)", path, len(text))
    print(text[:1500])

write_findings()